# The integrated tumor burden — a third TGI→OS bridge metric

A two-stage TGI→survival surrogate has a free choice of *which* on-treatment number drives the hazard.
v0.25 made that choice an explicit field (`structure.link_metric`) with two values: the default **week-8**
relative change (depth-only, blind to the tail) and **k_g**, the regrowth slope (tail-only, blind to depth).

This notebook adds the natural third option — the **integrated tumor burden** `log_burden_auc`, the
time-averaged log relative tumor size over the horizon (the AUC of the log-size curve) — the one summary
that integrates *both* depth and tail. The finding: it ranks the models a **third, distinct** way, and it
repairs a depth-blind pathology of the pure-tail k_g metric. *Population/trial level only; values illustrative.*

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.metrics import extract_tgi_metrics, _BURDEN_FLOOR

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
t = np.linspace(0.0, 260.0, 521)
WEEK8 = 'survival_link.nsclc_os_week8'
KG    = 'survival_link.nsclc_os_growth_rate'
BURDEN = 'survival_link.nsclc_os_burden_auc'
models = {
    'Claret (phenom.)':            'resistance.claret_2009.tgi',
    'two-population':              'resistance.nsclc_first_line.two_population',
    'Norton-Simon (complete)':     'drug_effect.norton_simon.nsclc',
    'Wang biexp (minimal resp.)':  'tgi_metrics.wang_2009.biexponential',
    'acquired resistance':         'resistance.nsclc_first_line.acquired',
}

## The metric is a closed-form trajectory summary

`log_burden_auc = (1/T) ∫ log(max(v/y0, floor)) dt`. A tumor held at baseline integrates to 0; a tumor held
at `c·y0` integrates to `log(c)`; eradication is floored at the detection limit, so the integral is stable
(never −∞).

In [ ]:
flat   = extract_tgi_metrics(t, np.full_like(t, 5.0), y0=5.0)['log_burden_auc']
half   = extract_tgi_metrics(t, np.full_like(t, 2.5), y0=5.0)['log_burden_auc']
erad   = extract_tgi_metrics(t, np.zeros_like(t),     y0=5.0)['log_burden_auc']
print(f'baseline (v=y0)      burden = {flat:+.3f}   (expect 0)')
print(f'held at half (v=y0/2) burden = {half:+.3f}   (expect log 0.5 = {np.log(0.5):+.3f})')
print(f'eradication (v=0)    burden = {erad:+.3f}   (floored at log {_BURDEN_FLOOR} = {np.log(_BURDEN_FLOOR):+.3f})')

## Each bridge metric ranks the models a different way

Median OS under each of the three links. Watch the **complete responder** (week-8 buries it mid-pack; both
tail-aware metrics rank it first) and the **minimal responder Wang** (k_g ranks it 2nd on a slow regrowth
slope; the depth-aware burden metric ranks it last).

In [ ]:
def mos(rid, link):
    return onkos.simulate(ds, rid, context=ctx, drug_effect=1.0, t=t, survival_link=link).median_os

print(f"{'model':28s} {'week-8':>7} {'k_g':>6} {'burden':>7}")
rows = {}
for name, rid in models.items():
    w, k, b = mos(rid, WEEK8), mos(rid, KG), mos(rid, BURDEN)
    rows[name] = (w, k, b)
    print(f'{name:28s} {w:7.0f} {k:6.0f} {b:7.0f}')

def order(i):
    return ' > '.join(sorted(rows, key=lambda n: rows[n][i], reverse=True))
print()
print('week-8 :', order(0))
print('k_g    :', order(1))
print('burden :', order(2))

## Burden repairs k_g's depth-blindness

`k_g` reads only the terminal regrowth slope, so it cannot tell that **Wang never shrank** (its nadir is
~75% of baseline). The integrated burden weighs depth, so the never-responding tumor lands last — where a
clinician would put it.

In [ ]:
for name in ('Wang biexp (minimal resp.)', 'two-population', 'Norton-Simon (complete)'):
    tr = onkos.simulate(ds, models[name], context=ctx, drug_effect=1.0, t=t)
    print(f"{name:28s} nadir/y0 = {tr.tumor_size.min()/tr.tumor_size[0]:.2f}   "
          f"burden = {tr.metrics['log_burden_auc']:+.2f}")
from onkos.budget import eligible_survival_links
print()
print('NSCLC/first eligible OS links:', len(eligible_survival_links(ds, ctx, 'OS')))

## The takeaway

Even the metric that uses the *whole* trajectory is still a **choice**, and it disagrees with both extremes —
so 'which bridge metric' remains a live model-selection axis. Onkos does not crown a winner: it makes the
metric a declared, swappable field and shows the ranking risk a silent choice carries. The new link is
`default=false`, so every default view and export is byte-identical. **No individual prediction, no therapy
ranking — the metric ranks models under a context, nothing more.**